# BipedalWalker-Hardcore TD3 — Colab

로컬에 GPU가 없어서, 결국 이 프로젝트의 학습·시연·재평가는 대부분 Colab에서 돌렸다. 이 노트북은 그 과정을 처음 받아보는 사람도 그대로 따라할 수 있게 순서대로 정리한 것이다. 저장소를 클론하는 것부터 시작해서, 다섯 개 모델 중 최종 후보인 모델1을 재생해보고, 실제로 학습을 다시 돌려보고, 100 에피소드로 재평가해서 README에 적힌 수치가 그대로 나오는지까지 확인한다. 위에서부터 순서대로 실행하면 된다.

## 1) 저장소 클론



In [6]:
!git clone https://github.com/AkimaraYouki/Summercamp.git
%cd Summercamp
!ls

Cloning into 'Summercamp'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 147 (delta 31), reused 140 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 29.38 MiB | 19.92 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/Summercamp
bipedalwalker-td3  COLAB_실행_가이드.md  README.md


`bipedalwalker-td3/` 안에 `src/`, `models/`, `results/`, `reports/` 네 개 폴더가 보이면 제대로 클론된 것이다.

## 2) 의존성 설치

In [7]:
!apt-get update -qq && apt-get install -y xvfb > /dev/null
!pip install -q pyvirtualdisplay
!pip install -q -r bipedalwalker-td3/requirements.txt

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 107.7 MB/s eta 0:00:00


여기서 xvfb를 같이 깔아주는 이유는, 원래 이 코드가 로컬에서 화면에 창을 띄우면서 보행 모습을 보여주는 방식이라 Colab처럼 화면이 없는 환경에서는 가짜 디스플레이가 있어야 그 창을 띄울 수 있기 때문이다. 학습이나 100ep 재평가는 화면을 안 띄우니 이 부분 없이도 돌아간다.

## 3) 가상 디스플레이 켜기 (세션당 한 번만, 시연 재생할 때만 필요)

In [8]:
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1400, 900))
display.start()

실행하면 `<pyvirtualdisplay.display.Display at 0x...>`라고 뜨는데, 이건 에러가 아니라 그냥 방금 만든 디스플레이 객체를 화면에 보여주는 것뿐이다. 가상 디스플레이가 잘 켜졌다는 뜻이니 그대로 다음으로 넘어가면 된다.

## 4) 모델1 시연 + 녹화

다섯 모델 중 낙상률이 가장 낮아서 최종 후보로 고른 모델1(`g2-si-speed-s201`)을 재생해본다. 기본 `run_play.sh`에는 녹화 옵션이 빠져 있어서, 여기서는 영상까지 남기려고 `video_record_dir`을 직접 붙여서 실행한다.

In [ ]:
%cd /content/Summercamp/bipedalwalker-td3/models/model1_g2-si-speed-s201
!python3 ../../src/Bipedalwalker_TD3_live.py --play 3 --ckpt ep794 --set \
  run_tag=g2-si-speed-s201 hardcore=true frame_skip=2 fall_penalty=-10.0 \
  activation=gelu final_layer_init_scale=0.003 critic_output_scale=10.0 \
  video_record_dir=/content/demo_videos

In [ ]:
from IPython.display import Video
Video("/content/demo_videos/g2-si-speed-s201_ep1.mp4", embed=True)

콘솔에 `[평가] Episode N: Reward = ...`가 세 번 찍히고 `/content/demo_videos/`에 mp4가 세 개 생겼으면 잘 된 것이다. 다른 모델을 보고 싶으면 `--ckpt`와 `--set`의 `run_tag`, 트릭 값만 해당 모델 폴더의 `run_train.sh`에 적힌 값으로 바꿔주면 된다(예를 들어 모델2는 `--ckpt ep3081 run_tag=scratch-initscale ...`).

## 5) 학습 재현

실제로 저 체크포인트가 어떻게 만들어졌는지 궁금하면 이 셀로 직접 학습을 다시 돌려볼 수 있다. 다만 시간이 꽤 걸리니, 끝까지 다 안 돌리고 로그만 봐도 충분하다.

In [12]:
#부모 채크포인트 복사
%cd /content/Summercamp/bipedalwalker-td3/models/model1_g2-si-speed-s201
!mkdir -p checkpoints/bipedalwalker-td3-hardcore-g2-si-speed-s201
!cp ../model2_scratch-initscale/checkpoints/bipedalwalker-td3-hardcore-scratch-initscale/ep3081_* \
    checkpoints/bipedalwalker-td3-hardcore-g2-si-speed-s201/
!ls checkpoints/bipedalwalker-td3-hardcore-g2-si-speed-s201/

/content/Summercamp/bipedalwalker-td3/models/model1_g2-si-speed-s201
ep3081_actor.pt    ep3081_critic2.pt  ep794_critic1.pt
ep3081_critic1.pt  ep794_actor.pt     ep794_critic2.pt


In [11]:
%cd /content/Summercamp/bipedalwalker-td3/models/model1_g2-si-speed-s201
!bash run_train.sh

/content/Summercamp/bipedalwalker-td3/models/model1_g2-si-speed-s201
[CPU 사용] torch.cuda.is_available() = False
[resume] 체크포인트 'ep3081' 가중치를 불러와서 이어서 학습합니다 (리플레이 버퍼/에피소드 카운터는 새로 시작)
2026-07-14 15:12:04.846206: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-14 15:12:05.652339: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
TensorBoard: tensorboard/td3_bipedalwalker-td3-hardcore-g2-si-speed-s201_20260714-151210  (tensorboard --logdir tensorboard)
TD3 학습 시작! device=cpu, state_dim=24, action_dim=4, hardcore=True, hidden_dim=256, num_hidden_layers=2, activation=gelu
[hardcore 트릭] frame_skip=2, fall_penalty=-10.0
[안정화 트릭] use_layer_norm=False, final_layer_init_scale=0.003, critic

매 에피소드마다 `Episode [N/2000] Reward: ... | 최근 100개 평균: ...`이 찍히다가, 최근 100개 평균이 300을 넘는 순간 `환경을 풀었습니다!`라는 메시지와 함께 멈춘다. 굳이 그 순간까지 기다리지 않아도, 최근 100개 평균이 꾸준히 올라가는 걸 확인하는 것만으로도 이 학습이 실제로 재현된다는 건 충분히 알 수 있다.

## 6) 100ep 고정 시드 재평가

학습 중 기록되는 값과 별개로, 노이즈 없이 같은 지형 100개에서 다시 평가한 수치가 README에 적힌 해결률·낙상률이다. 이 셀을 돌리면 그 수치가 그대로 재현되는지 확인할 수 있다.

In [ ]:
%cd /content/Summercamp/bipedalwalker-td3/models/model1_g2-si-speed-s201
!bash run_eval100.sh

아래와 비슷하게 나오면 맞는 것이다. README "결과" 표에 있는 모델1 값(해결률 83%, 낙상률 12%, 비낙상 평균 304.73, 클리어 502.6)과 같아야 한다.

```
[100ep 재평가] run_tag=g2-si-speed-s201 ckpt=ep794
  해결률(reward>=300): 83.0%
  낙상률: 12.0%
  비낙상 평균 reward: 304.73
  평균 클리어 step: 502.6
```

## 7) 알고리즘 비교 파이프라인 (TD3 vs SAC/PPO/DDPG, 선택)

여기부터는 5개 모델과는 별개로, 애초에 왜 TD3를 골랐는지를 뒷받침하려고 만든 비교 실험이다. `config.TOTAL_STEPS`가 2,000,000이라 시간이 오래 걸리니, 이 세션 안에서 끝까지 돌릴 생각은 안 해도 된다. 더 자세한 내용은 `algo_comparison/README.md`에 적어뒀다.

In [ ]:
%cd /content/Summercamp/bipedalwalker-td3/algo_comparison
!pip install -q -r requirements.txt
!python3 run_td3.py --seed 0

이 스크립트는 아직 200만 step에서 자동으로 멈추는 로직이 없어서, 어느 정도 돌아가는 걸 확인했으면 직접 중단시켜야 한다.

In [ ]:
!python3 run_baselines.py --algo sac --seed 0

이쪽은 Stable-Baselines3를 그대로 쓰기 때문에 정해진 step에서 알아서 멈춘다. 둘 다 오래 걸리는 스크립트라, 끝까지 도는 걸 보기보다는 학습 로그가 정상적으로 찍히기 시작하는지만 확인해도 코드가 제대로 동작한다는 건 알 수 있다. 실제 비교 수치는 다음 단계에서 원본 데이터로 바로 확인한다.

## 8) 알고리즘 비교 원본 데이터 검증

In [ ]:
%cd /content/Summercamp/bipedalwalker-td3/reports/figures
!python3 summarize_1M_step.py

이 스크립트는 슬라이드에 있는 "1M step 지점 값" 표를 원본 CSV에서 그대로 다시 계산해서 보여준다. `reports/figures/README.md`의 검증 결과 표와 같은 값이 나오면 된다. 참고로 팀원 데이터 쪽 세 개(TD3, PPO, DDPG)는 슬라이드에 적힌 값과 다르게 나오는데, 이건 버그가 아니라 원본 데이터를 검증하다가 직접 발견한 불일치라서 같은 README에 원인까지 적어뒀다.

## 9) 결과물 다운로드

여기까지 돌리면서 나온 체크포인트와 결과물을 로컬로 다시 가져오고 싶으면 아래 셀로 압축해서 받으면 된다.

In [ ]:
%cd /content/Summercamp
!zip -r results_from_colab.zip bipedalwalker-td3/models/*/checkpoints bipedalwalker-td3/results

from google.colab import files
files.download("/content/Summercamp/results_from_colab.zip")